In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, ttest_ind, mannwhitneyu, f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
import warnings
warnings.filterwarnings('ignore')

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Advanced Statistical Analysis for Cardiovascular Risk Factors")
print("=" * 80)
print("Md. Faishal Ahmed Rudro")
print("Email: md.faishalrudro@gmail.com")
print("=" * 80)

# Load the dataset
df = pd.read_excel("/kaggle/input/testdatasetmain/test-dataset.xlsx")
print(f"Dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns.")

# Display initial dataset information
print("\n1. Dataset Overview:")
print("-" * 50)
print(f"Number of samples: {df.shape[0]}")
print(f"Number of features: {df.shape[1]}")
print(f"Missing values: {df.isnull().sum().sum()}")
print("\nTarget variable distribution (BP Status):")
if 'RESULT_STAT_BP' in df.columns:
    print(df['RESULT_STAT_BP'].value_counts())
else:
    print("BP Status variable not found in dataset.")

# Identify variable types
print("\n2. Variable Identification:")
print("-" * 50)
# Exclude identifier and obviously non-predictive columns
excluded_cols = [
    'Unnamed: 0', 'household_id', 'user_id', 'profile_name', 'father_name', 'mother_name',
    'birthday', 'TAG_NAME'
]

# Identify numerical and categorical features
numeric_features = []
categorical_features = []

for col in df.columns:
    if col in excluded_cols:
        continue
    
    if df[col].dtype in ['int64', 'float64']:
        if col not in ['SYSTOLIC', 'DIASTOLIC', 'RESULT_STAT_BP']:
            numeric_features.append(col)
    elif df[col].dtype == 'object':
        if len(df[col].unique()) < df.shape[0] * 0.5:  # Avoid columns with too many unique values
            categorical_features.append(col)

print(f"Numerical features ({len(numeric_features)}): {numeric_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"Target variables: SYSTOLIC, DIASTOLIC, RESULT_STAT_BP")

# Clean dataset for analysis - handle missing values
print("\n3. Data Preparation:")
print("-" * 50)
# For BP analysis
bp_df = df.dropna(subset=['RESULT_STAT_BP'])
print(f"Samples after removing missing BP status: {bp_df.shape[0]}")

# Function to perform Chi-square test for categorical variables
def chi_square_test(df, feature, target, alpha=0.05):
    contingency_table = pd.crosstab(df[feature], df[target])
    if contingency_table.shape[0] < 2 or contingency_table.shape[1] < 2:
        return feature, None, None, None, "Insufficient categories"
    
    # Check for expected frequencies less than 5
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    
    # Check if more than 20% of cells have expected frequencies < 5
    expected_lt_5 = (expected < 5).sum()
    total_cells = expected.size
    percent_lt_5 = (expected_lt_5 / total_cells) * 100
    
    note = ""
    if percent_lt_5 > 20:
        note = f"Warning: {percent_lt_5:.1f}% of cells have expected frequency < 5"
    
    significance = "Significant" if p < alpha else "Not significant"
    return feature, p, contingency_table.shape, significance, note

# Function for numerical feature analysis with t-test or Mann-Whitney U test
def analyze_numerical_feature(df, feature, target, alpha=0.05):
    # Group data
    groups = []
    group_names = []
    
    for group_name in df[target].unique():
        group = df[df[target] == group_name][feature].dropna()
        if len(group) > 0:
            groups.append(group)
            group_names.append(group_name)
    
    if len(groups) < 2:
        return feature, None, "Insufficient data"
    
    # If binary target and two groups
    if len(groups) == 2:
        # Check normality (simplified approach - for larger datasets consider Shapiro-Wilk)
        try:
            # Use Mann-Whitney U test (non-parametric, doesn't assume normality)
            stat, p = mannwhitneyu(groups[0], groups[1])
            test_name = "Mann-Whitney U"
        except:
            # Fallback to t-test if Mann-Whitney fails
            stat, p = ttest_ind(groups[0], groups[1], equal_var=False)
            test_name = "Welch's t-test"
    else:
        # Use ANOVA for more than two groups
        stat, p = f_oneway(*groups)
        test_name = "ANOVA"
    
    # Calculate means for each group
    means = [group.mean() for group in groups]
    means_info = dict(zip(group_names, means))
    
    significance = "Significant" if p < alpha else "Not significant"
    return feature, p, significance, test_name, means_info

print("\n4. Statistical Analysis:")
print("-" * 50)
print("\na) Categorical Variables Analysis with Chi-square Tests:")
bp_categorical_results = []

for feature in categorical_features:
    if feature in bp_df.columns:
        result = chi_square_test(bp_df, feature, 'RESULT_STAT_BP')
        if result[1] is not None:
            bp_categorical_results.append(result)

# Sort by p-value
bp_categorical_results.sort(key=lambda x: x[1] if x[1] is not None else 1)

# Display results
for feature, p_val, shape, significance, note in bp_categorical_results:
    if p_val is not None:
        print(f"Feature: {feature}, p-value: {p_val:.5f}, Table Shape: {shape}, {significance}")
        if note:
            print(f"   Note: {note}")

print("\nb) Numerical Variables Analysis:")
bp_numerical_results = []

for feature in numeric_features:
    if feature in bp_df.columns:
        result = analyze_numerical_feature(bp_df, feature, 'RESULT_STAT_BP')
        if result[1] is not None:
            bp_numerical_results.append(result)

# Sort by p-value
bp_numerical_results.sort(key=lambda x: x[1] if x[1] is not None else 1)

# Display results
for feature, p_val, significance, test_name, means_info in bp_numerical_results:
    if p_val is not None:
        print(f"Feature: {feature}, p-value: {p_val:.5f}, {significance}, Test: {test_name}")
        print(f"   Group means: {means_info}")

# Feature importance using Random Forest for BP classification
print("\n5. Feature Importance Analysis:")
print("-" * 50)

def get_feature_importance(df, target, features):
    # Prepare data
    X = df[features].copy()
    
    # Handle categorical features
    for col in X.select_dtypes(include=['object']).columns:
        X[col] = pd.factorize(X[col])[0]
    
    # Fill missing values with median for numeric and mode for categorical
    for col in X.columns:
        if X[col].dtype in ['int64', 'float64']:
            X[col] = X[col].fillna(X[col].median())
        else:
            X[col] = X[col].fillna(X[col].mode()[0])
    
    y = df[target]
    
    # Handle target variable if needed
    if y.dtype == 'object':
        y = pd.factorize(y)[0]
    
    # Train a Random Forest classifier
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X, y)
    
    # Get feature importances
    importances = clf.feature_importances_
    
    # Create a DataFrame of feature importances
    feature_importance_df = pd.DataFrame({
        'Feature': features,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    return feature_importance_df

# Prepare features for importance analysis
all_features = numeric_features + categorical_features
bp_features = [f for f in all_features if f in bp_df.columns]

# Handle categorical features
bp_df_processed = bp_df.copy()
for col in bp_df_processed.select_dtypes(include=['object']).columns:
    if col in bp_features:
        bp_df_processed[col] = pd.factorize(bp_df_processed[col])[0]

# Fill missing values
for col in bp_features:
    if bp_df_processed[col].dtype in ['int64', 'float64']:
        bp_df_processed[col] = bp_df_processed[col].fillna(bp_df_processed[col].median())
    else:
        bp_df_processed[col] = bp_df_processed[col].fillna(bp_df_processed[col].mode()[0])

# Initialize bp_importance with empty DataFrame in case the try block fails
bp_importance = pd.DataFrame(columns=['Feature', 'Importance'])

# Get feature importance for BP
try:
    bp_importance = get_feature_importance(bp_df_processed, 'RESULT_STAT_BP', bp_features)
    print("Feature importance for BP status prediction:")
    print(bp_importance.head(15))
except Exception as e:
    print(f"Error in feature importance analysis: {e}")
    # Ensure bp_importance exists even if there's an error
    if 'bp_importance' not in locals() or bp_importance is None:
        bp_importance = pd.DataFrame(columns=['Feature', 'Importance'])

# Correlation analysis for numerical features
print("\n6. Correlation Analysis for Numerical Features:")
print("-" * 50)
try:
    numerical_df = bp_df[['SYSTOLIC', 'DIASTOLIC'] + numeric_features].copy()
    correlation_matrix = numerical_df.corr()
    
    print("Key correlations with BP measurements:")
    systolic_corr = correlation_matrix['SYSTOLIC'].sort_values(ascending=False)
    diastolic_corr = correlation_matrix['DIASTOLIC'].sort_values(ascending=False)
    
    print("\nTop correlations with SYSTOLIC BP:")
    print(systolic_corr.head(6))
    
    print("\nTop correlations with DIASTOLIC BP:")
    print(diastolic_corr.head(6))
except Exception as e:
    print(f"Error in correlation analysis: {e}")

print("\n7. Statistical Summary and Conclusions:")
print("-" * 50)
print("\nSignificant factors associated with BP status (p < 0.05):")

print("\nCategorical factors:")
significant_categorical = [feat for feat, p_val, _, sig, _ in bp_categorical_results if p_val is not None and p_val < 0.05]
if significant_categorical:
    for feat in significant_categorical:
        print(f"- {feat}")
else:
    print("No statistically significant categorical features found")

print("\nNumerical factors:")
significant_numerical = [feat for feat, p_val, sig, _, _ in bp_numerical_results if p_val is not None and p_val < 0.05]
if significant_numerical:
    for feat in significant_numerical:
        print(f"- {feat}")
else:
    print("No statistically significant numerical features found")

# Final conclusion
print("\nConclusion:")
if len(significant_categorical) > 0 or len(significant_numerical) > 0:
    print("Statistical analysis reveals significant relationships between several factors and BP status.")
    print("These factors appear to be strongly associated with blood pressure classification and could be")
    print("important for predicting cardiovascular risk, similar to findings in the Kaunas cohort study.")
    
    # Now bp_importance is guaranteed to exist
    if not bp_importance.empty:
        top_features = bp_importance['Feature'].head(3).tolist()
        print(f"\nThe most predictive features based on machine learning importance scores were:")
        for feat in top_features:
            print(f"- {feat}")
    else:
        print("\nFeature importance analysis did not produce valid results.")
else:
    print("No statistically significant relationships were found between the examined factors and BP status.")
    print("This differs from the Kaunas cohort study findings, which found strong relationships between")
    print("childhood anthropometric measurements and adult cardiovascular risk factors.")

print("\nMethodological Note:")
print("This analysis employed Chi-square tests for categorical variables and Mann-Whitney U/ANOVA tests")
print("for numerical variables to identify statistically significant associations with BP status.")
print("Additionally, Random Forest-based feature importance analysis was used to identify the most")
print("predictive factors in a multivariate context, accounting for potential interactions between variables.")

Advanced Statistical Analysis for Cardiovascular Risk Factors
Md. Faishal Ahmed Rudro
Email: md.faishalrudro@gmail.com
Dataset loaded with 29999 rows and 34 columns.

1. Dataset Overview:
--------------------------------------------------
Number of samples: 29999
Number of features: 34
Missing values: 333065

Target variable distribution (BP Status):
RESULT_STAT_BP
Normal             11490
Prehypertension     7059
Mild High           3574
Low                 2704
Moderate High       1598
Severe High          630
High                 545
Name: count, dtype: int64

2. Variable Identification:
--------------------------------------------------
Numerical features (12): ['age', 'is_poor', 'is_freedom_fighter', 'had_stroke', 'has_cardiovascular_disease', 'HEIGHT', 'WEIGHT', 'BMI', 'SUGAR', 'PULSE_RATE', 'SPO2', 'MUAC']
Categorical features (10): ['total_income', 'union_name', 'gender', 'disabilities_name', 'RESULT_STAT_BP', 'RESULT_STAT_BMI', 'RESULT_STAT_SUGAR', 'RESULT_STAT_PR', 'RESULT_ST

In [4]:
# 📦 Imports
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, f_oneway
import warnings
warnings.filterwarnings('ignore')

# 📥 Load dataset
# Upload your file in Kaggle sidebar (left > "Upload")
df = pd.read_excel("/kaggle/input/testdatasetmain/test-dataset.xlsx")

# 🎯 Set your target variable
target = 'RESULT_STAT_BP'

# 🧹 Drop rows with missing target
df = df.dropna(subset=[target])

# 🔍 Identify categorical and numerical columns
categorical_cols = df.select_dtypes(include='object').columns.tolist()
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# ❌ Columns to ignore (names, IDs, other targets)
exclude_cols = ['profile_name', 'father_name', 'mother_name', 'TAG_NAME',
                'RESULT_STAT_BMI', 'RESULT_STAT_SUGAR', 'RESULT_STAT_PR',
                'RESULT_STAT_SPO2', 'RESULT_STAT_MUAC', 'Unnamed: 0',
                'user_id', 'household_id']

categorical_cols = [col for col in categorical_cols if col not in exclude_cols]
numerical_cols = [col for col in numerical_cols if col not in exclude_cols]

# 📊 Run Chi-Square and ANOVA
results = []

# Chi-Square for Categorical
for col in categorical_cols:
    if df[col].nunique() > 1:
        try:
            table = pd.crosstab(df[col], df[target])
            chi2, p, _, _ = chi2_contingency(table)
            results.append((col, 'categorical', p))
        except:
            continue

# ANOVA for Numerical
for col in numerical_cols:
    try:
        grouped = [group[col].dropna() for name, group in df.groupby(target)]
        if len(grouped) >= 2 and all(len(g) > 1 for g in grouped):
            f_stat, p = f_oneway(*grouped)
            results.append((col, 'numerical', p))
    except:
        continue

# 📈 Filter & show significant results
significant_results = [(col, typ, p) for col, typ, p in results if p < 0.05]
significant_results = sorted(significant_results, key=lambda x: x[2])

# 📋 Display
print("📌 Significant Features Related to RESULT_STAT_BP:\n")
for col, typ, p in significant_results:
    print(f"{col:<20} | Type: {typ:<11} | p-value: {p:.4e}")


📌 Significant Features Related to RESULT_STAT_BP:

RESULT_STAT_BP       | Type: categorical | p-value: 0.0000e+00
age                  | Type: numerical   | p-value: 0.0000e+00
SYSTOLIC             | Type: numerical   | p-value: 0.0000e+00
DIASTOLIC            | Type: numerical   | p-value: 0.0000e+00
union_name           | Type: categorical | p-value: 8.4501e-242
gender               | Type: categorical | p-value: 2.8189e-80
birthday             | Type: categorical | p-value: 3.8612e-51
PULSE_RATE           | Type: numerical   | p-value: 1.2261e-27
total_income         | Type: categorical | p-value: 1.1736e-24
WEIGHT               | Type: numerical   | p-value: 1.7674e-05
BMI                  | Type: numerical   | p-value: 1.0707e-04
SPO2                 | Type: numerical   | p-value: 3.1714e-03


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, ttest_ind, mannwhitneyu, f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.4f}'.format)

print("Advanced Statistical Analysis for Cardiovascular Risk Factors")
print("=" * 80)
print("Md. Faishal Ahmed Rudro")
print("Email: md.faishalrudro@gmail.com")
print("=" * 80)

# Load the dataset
df = pd.read_excel("/kaggle/input/testdatasetmain/test-dataset.xlsx")
print(f"Dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns.")

# Display initial dataset information
print("\n1. Dataset Overview:")
print("-" * 50)
print(f"Number of samples: {df.shape[0]}")
print(f"Number of features: {df.shape[1]}")
print(f"Missing values: {df.isnull().sum().sum()}")
print("\nTarget variable distribution (BP Status):")
if 'RESULT_STAT_BP' in df.columns:
    print(df['RESULT_STAT_BP'].value_counts())
    print("Percentage distribution:")
    print(df['RESULT_STAT_BP'].value_counts(normalize=True).apply(lambda x: f"{x*100:.2f}%"))
else:
    print("BP Status variable not found in dataset.")

# Identify variable types automatically
print("\n2. Variable Identification:")
print("-" * 50)
# Exclude identifier and obviously non-predictive columns
excluded_cols = [
    'Unnamed: 0', 'household_id', 'user_id', 'profile_name', 'father_name', 'mother_name',
    'birthday', 'TAG_NAME'
]

# Identify numerical and categorical features automatically
numeric_features = []
categorical_features = []

for col in df.columns:
    if col in excluded_cols or col in ['SYSTOLIC', 'DIASTOLIC', 'RESULT_STAT_BP']:
        continue
    
    if df[col].dtype in ['int64', 'float64']:
        numeric_features.append(col)
    elif df[col].dtype == 'object':
        if len(df[col].unique()) < df.shape[0] * 0.5:  # Avoid columns with too many unique values
            categorical_features.append(col)

print(f"Numerical features ({len(numeric_features)}):")
for i, feat in enumerate(numeric_features):
    print(f"{i+1}. {feat}")

print(f"\nCategorical features ({len(categorical_features)}):")
for i, feat in enumerate(categorical_features):
    print(f"{i+1}. {feat}")

print(f"\nTarget variables: SYSTOLIC, DIASTOLIC, RESULT_STAT_BP")

# Clean dataset for analysis - handle missing values
print("\n3. Data Preparation:")
print("-" * 50)
# For BP analysis
bp_df = df.dropna(subset=['RESULT_STAT_BP'])
print(f"Samples after removing missing BP status: {bp_df.shape[0]}")

# Missing value summary
print("\nMissing values per feature (top 10):")
missing_summary = df.isnull().sum().sort_values(ascending=False)
print(missing_summary[missing_summary > 0].head(10))

# Function to perform Chi-square test for categorical variables
def chi_square_test(df, feature, target, alpha=0.05):
    # Create contingency table
    contingency_table = pd.crosstab(df[feature], df[target])
    
    # Skip if not enough categories
    if contingency_table.shape[0] < 2 or contingency_table.shape[1] < 2:
        return feature, None, None, None, "Insufficient categories"
    
    # Perform chi-square test
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    
    # Check if more than 20% of cells have expected frequencies < 5
    expected_lt_5 = (expected < 5).sum()
    total_cells = expected.size
    percent_lt_5 = (expected_lt_5 / total_cells) * 100
    
    note = ""
    if percent_lt_5 > 20:
        note = f"Warning: {percent_lt_5:.1f}% of cells have expected frequency < 5"
    
    # Effect size (Cramer's V)
    n = contingency_table.sum().sum()
    min_dim = min(contingency_table.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0
    
    significance = "Significant" if p < alpha else "Not significant"
    return feature, p, contingency_table.shape, significance, note, cramers_v

# Function for numerical feature analysis with t-test or Mann-Whitney U test
def analyze_numerical_feature(df, feature, target, alpha=0.05):
    # Group data
    groups = []
    group_names = []
    
    for group_name in df[target].unique():
        group = df[df[target] == group_name][feature].dropna()
        if len(group) > 0:
            groups.append(group)
            group_names.append(group_name)
    
    if len(groups) < 2:
        return feature, None, "Insufficient data", None, None, None
    
    # If binary target and two groups
    if len(groups) == 2:
        # Use Mann-Whitney U test (non-parametric, doesn't assume normality)
        try:
            stat, p = mannwhitneyu(groups[0], groups[1])
            test_name = "Mann-Whitney U"
            
            # Calculate effect size (r = Z / sqrt(N))
            n1, n2 = len(groups[0]), len(groups[1])
            z = stat / np.sqrt((n1 * n2 * (n1 + n2 + 1)) / 6)
            effect_size = abs(z) / np.sqrt(n1 + n2)
            
        except Exception as e:
            # Fallback to t-test if Mann-Whitney fails
            stat, p = ttest_ind(groups[0], groups[1], equal_var=False)
            test_name = "Welch's t-test"
            
            # Calculate Cohen's d
            mean1, mean2 = np.mean(groups[0]), np.mean(groups[1])
            sd1, sd2 = np.std(groups[0], ddof=1), np.std(groups[1], ddof=1)
            pooled_sd = np.sqrt(((n1 - 1) * sd1**2 + (n2 - 1) * sd2**2) / (n1 + n2 - 2))
            effect_size = abs(mean1 - mean2) / pooled_sd
    else:
        # Use ANOVA for more than two groups
        stat, p = f_oneway(*groups)
        test_name = "ANOVA"
        
        # Calculate eta squared for ANOVA
        group_means = [np.mean(group) for group in groups]
        grand_mean = np.mean([val for group in groups for val in group])
        between_ss = sum(len(group) * (mean - grand_mean)**2 for group, mean in zip(groups, group_means))
        total_ss = sum((val - grand_mean)**2 for group in groups for val in group)
        effect_size = between_ss / total_ss if total_ss > 0 else 0
    
    # Calculate means for each group
    means = [group.mean() for group in groups]
    means_info = dict(zip(group_names, means))
    
    significance = "Significant" if p < alpha else "Not significant"
    return feature, p, significance, test_name, means_info, effect_size

print("\n4. Statistical Analysis:")
print("-" * 50)
print("\na) Categorical Variables Analysis with Chi-square Tests:")
print("   Analyzing relationship between categorical variables and BP status")

bp_categorical_results = []

for feature in categorical_features:
    if feature in bp_df.columns:
        result = chi_square_test(bp_df, feature, 'RESULT_STAT_BP')
        if result[1] is not None:
            bp_categorical_results.append(result)

# Sort by p-value
bp_categorical_results.sort(key=lambda x: x[1] if x[1] is not None else 1)

# Display results in tabular format
print("\n   Results sorted by statistical significance (p-value):")
print("   {:<30} {:<12} {:<25} {:<15} {:<10}".format("Feature", "p-value", "Table Shape", "Significance", "Effect Size"))
print("   " + "-" * 90)

for feature, p_val, shape, significance, note, effect_size in bp_categorical_results:
    if p_val is not None:
        print("   {:<30} {:<12.5f} {:<25} {:<15} {:<10.3f}".format(
            feature, p_val, str(shape), significance, effect_size))
        if note:
            print(f"   Note: {note}")

print("\nb) Numerical Variables Analysis:")
print("   Analyzing relationship between numerical variables and BP status")

bp_numerical_results = []

for feature in numeric_features:
    if feature in bp_df.columns:
        result = analyze_numerical_feature(bp_df, feature, 'RESULT_STAT_BP')
        if result[1] is not None:
            bp_numerical_results.append(result)

# Sort by p-value
bp_numerical_results.sort(key=lambda x: x[1] if x[1] is not None else 1)

# Display results in tabular format
print("\n   Results sorted by statistical significance (p-value):")
print("   {:<30} {:<12} {:<15} {:<15} {:<10}".format("Feature", "p-value", "Significance", "Test", "Effect Size"))
print("   " + "-" * 90)

for feature, p_val, significance, test_name, means_info, effect_size in bp_numerical_results:
    if p_val is not None:
        print("   {:<30} {:<12.5f} {:<15} {:<15} {:<10.3f}".format(
            feature, p_val, significance, test_name, effect_size))
        print("   Group means: " + ", ".join([f"{k}: {v:.2f}" for k, v in means_info.items()]))

# Feature importance using Random Forest for BP classification
print("\n5. Feature Importance Analysis:")
print("-" * 50)

def get_feature_importance(df, target, features):
    # Prepare data
    X = df[features].copy()
    y = df[target].copy()
    
    # Skip features with too many missing values
    valid_features = []
    for col in X.columns:
        if X[col].isnull().sum() / len(X) < 0.5:  # Skip if >50% missing
            valid_features.append(col)
        else:
            print(f"   Skipping feature '{col}' due to excessive missing values ({X[col].isnull().sum()} out of {len(X)})")
    
    X = X[valid_features]
    
    # Replace infinite values with NaN
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # Create numerical and categorical feature lists for appropriate imputation
    numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
    
    # Build preprocessing pipeline
    preprocessor = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent'))  # Using most_frequent for both categorical and numerical
    ])
    
    # Apply preprocessing
    X_processed = preprocessor.fit_transform(X)
    
    # Convert back to DataFrame
    X_processed = pd.DataFrame(X_processed, columns=X.columns)
    
    # Handle target variable if needed
    if y.dtype == 'object':
        y = pd.factorize(y)[0]
    
    # Train a Random Forest classifier
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    
    try:
        clf.fit(X_processed, y)
        
        # Get feature importances
        importances = clf.feature_importances_
        
        # Create a DataFrame of feature importances
        feature_importance_df = pd.DataFrame({
            'Feature': X.columns,
            'Importance': importances
        }).sort_values('Importance', ascending=False)
        
        return feature_importance_df
    
    except Exception as e:
        print(f"   Error during model fitting: {e}")
        return pd.DataFrame(columns=['Feature', 'Importance'])

print("   Running Random Forest feature importance analysis for BP status prediction")
print("   This analysis helps identify which features are most predictive of BP status in a multivariate context\n")

# Prepare features for importance analysis
all_features = numeric_features + categorical_features
bp_features = [f for f in all_features if f in bp_df.columns]

try:
    # Get feature importance for BP
    bp_importance = get_feature_importance(bp_df, 'RESULT_STAT_BP', bp_features)
    
    if not bp_importance.empty:
        print("\n   Top 15 features by importance for BP status prediction:")
        print("   {:<30} {:<15}".format("Feature", "Importance Score"))
        print("   " + "-" * 45)
        for _, row in bp_importance.head(15).iterrows():
            print("   {:<30} {:<15.4f}".format(row['Feature'], row['Importance']))
    else:
        print("   Feature importance analysis did not produce valid results.")
        
except Exception as e:
    print(f"   Error in feature importance analysis: {e}")
    bp_importance = pd.DataFrame(columns=['Feature', 'Importance'])

# Correlation analysis for numerical features
print("\n6. Correlation Analysis for Numerical Features:")
print("-" * 50)
try:
    # Include only numeric features with <= 30% missing values
    valid_numeric = []
    for col in numeric_features:
        if col in bp_df.columns and bp_df[col].isnull().sum() / len(bp_df) <= 0.3:
            valid_numeric.append(col)
    
    numerical_df = bp_df[['SYSTOLIC', 'DIASTOLIC'] + valid_numeric].copy()
    
    # Impute missing values with median for correlation analysis
    for col in numerical_df.columns:
        if numerical_df[col].isnull().sum() > 0:
            numerical_df[col] = numerical_df[col].fillna(numerical_df[col].median())
    
    correlation_matrix = numerical_df.corr()
    
    print("   Key correlations with BP measurements:")
    systolic_corr = correlation_matrix['SYSTOLIC'].sort_values(ascending=False)
    diastolic_corr = correlation_matrix['DIASTOLIC'].sort_values(ascending=False)
    
    print("\n   Top correlations with SYSTOLIC BP:")
    print("   {:<30} {:<15}".format("Feature", "Correlation"))
    print("   " + "-" * 45)
    # Skip the self-correlation (SYSTOLIC with SYSTOLIC)
    for feature, corr in systolic_corr.iloc[1:7].items():
        print("   {:<30} {:<15.4f}".format(feature, corr))
    
    print("\n   Top correlations with DIASTOLIC BP:")
    print("   {:<30} {:<15}".format("Feature", "Correlation"))
    print("   " + "-" * 45)
    # Skip the self-correlation (DIASTOLIC with DIASTOLIC)
    for feature, corr in diastolic_corr.iloc[1:7].items():
        print("   {:<30} {:<15.4f}".format(feature, corr))
    
except Exception as e:
    print(f"   Error in correlation analysis: {e}")

print("\n7. Statistical Summary and Conclusions:")
print("-" * 50)
print("\nSignificant factors associated with BP status (p < 0.05):")

print("\nStatistically significant categorical factors:")
significant_categorical = [(feat, p_val, effect) for feat, p_val, _, sig, _, effect in bp_categorical_results 
                          if p_val is not None and p_val < 0.05]

if significant_categorical:
    print("   {:<30} {:<15} {:<15}".format("Feature", "p-value", "Effect Size"))
    print("   " + "-" * 60)
    for feat, p_val, effect in significant_categorical:
        print("   {:<30} {:<15.5f} {:<15.4f}".format(feat, p_val, effect))
else:
    print("   No statistically significant categorical features found")

print("\nStatistically significant numerical factors:")
significant_numerical = [(feat, p_val, effect) for feat, p_val, sig, _, _, effect in bp_numerical_results 
                        if p_val is not None and p_val < 0.05]

if significant_numerical:
    print("   {:<30} {:<15} {:<15}".format("Feature", "p-value", "Effect Size"))
    print("   " + "-" * 60)
    for feat, p_val, effect in significant_numerical:
        print("   {:<30} {:<15.5f} {:<15.4f}".format(feat, p_val, effect))
else:
    print("   No statistically significant numerical features found")

# Final conclusion
print("\nConclusion:")
if len(significant_categorical) > 0 or len(significant_numerical) > 0:
    print("   Statistical analysis reveals significant relationships between several factors and BP status.")
    print("   These factors appear to be strongly associated with blood pressure classification and could be")
    print("   important for predicting cardiovascular risk, similar to findings in the Kaunas cohort study.")
    
    if not bp_importance.empty:
        top_features = bp_importance['Feature'].head(5).tolist()
        print(f"\n   The most predictive features based on machine learning importance scores were:")
        print("   {:<30} {:<15}".format("Feature", "Importance"))
        print("   " + "-" * 45)
        for feat in top_features:
            importance = bp_importance[bp_importance['Feature'] == feat]['Importance'].values[0]
            print("   {:<30} {:<15.4f}".format(feat, importance))
    else:
        print("\n   Feature importance analysis did not produce valid results.")
else:
    print("   No statistically significant relationships were found between the examined factors and BP status.")
    print("   This differs from the Kaunas cohort study findings, which found strong relationships between")
    print("   childhood anthropometric measurements and adult cardiovascular risk factors.")

print("\nMethodological Note:")
print("   This analysis employed Chi-square tests for categorical variables and Mann-Whitney U/ANOVA tests")
print("   for numerical variables to identify statistically significant associations with BP status.")
print("   Effect sizes were calculated to quantify the strength of associations beyond statistical significance.")
print("   Additionally, Random Forest-based feature importance analysis was used to identify the most")
print("   predictive factors in a multivariate context, accounting for potential interactions between variables.")

Advanced Statistical Analysis for Cardiovascular Risk Factors
Md. Faishal Ahmed Rudro
Email: md.faishalrudro@gmail.com
Dataset loaded with 29999 rows and 34 columns.

1. Dataset Overview:
--------------------------------------------------
Number of samples: 29999
Number of features: 34
Missing values: 333065

Target variable distribution (BP Status):
RESULT_STAT_BP
Normal             11490
Prehypertension     7059
Mild High           3574
Low                 2704
Moderate High       1598
Severe High          630
High                 545
Name: count, dtype: int64
Percentage distribution:
RESULT_STAT_BP
Normal             41.63%
Prehypertension    25.58%
Mild High          12.95%
Low                 9.80%
Moderate High       5.79%
Severe High         2.28%
High                1.97%
Name: proportion, dtype: object

2. Variable Identification:
--------------------------------------------------
Numerical features (12):
1. age
2. is_poor
3. is_freedom_fighter
4. had_stroke
5. has_cardiovascu

In [4]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, f_oneway

# Load the dataset
df = pd.read_excel("/kaggle/input/testdatasetmain/test-dataset.xlsx")  # Adjust path as needed

# Drop unnecessary or ID-like columns
df = df.drop(columns=["Unnamed: 0", "user_id", "profile_name", "father_name", "mother_name"], errors="ignore")

# Remove rows where target is missing
target_col = "RESULT_STAT_BP"
df = df.dropna(subset=[target_col])

# Detect data types
categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Remove target from the analysis
if target_col in categorical_cols:
    categorical_cols.remove(target_col)
if target_col in numerical_cols:
    numerical_cols.remove(target_col)

# Chi-square test for categorical columns
chi2_results = []
for col in categorical_cols:
    try:
        table = pd.crosstab(df[col], df[target_col])
        if table.shape[0] > 1:
            chi2, p, _, _ = chi2_contingency(table)
            chi2_results.append((col, "categorical", p))
    except:
        pass

# ANOVA test for numerical columns
anova_results = []
for col in numerical_cols:
    try:
        groups = [g[col].dropna() for _, g in df.groupby(target_col)]
        if all(len(g) > 1 for g in groups):
            f, p = f_oneway(*groups)
            anova_results.append((col, "numerical", p))
    except:
        pass

# Combine and show
results = pd.DataFrame(chi2_results + anova_results, columns=["Feature", "Type", "p-value"])
results = results.sort_values("p-value")
display(results)


,Feature,Type,p-value
16,SYSTOLIC,numerical,0.0000
11,age,numerical,0.0000
17,DIASTOLIC,numerical,0.0000
1,union_name,categorical,0.0000
8,RESULT_STAT_PR,categorical,0.0000
3,gender,categorical,0.0000
2,birthday,categorical,0.0000
22,PULSE_RATE,numerical,0.0000
0,total_income,categorical,0.0000
19,WEIGHT,numerical,0.0000


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, ttest_ind, mannwhitneyu, f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.4f}'.format)

print("Advanced Statistical Analysis for Cardiovascular Risk Factors")
print("=" * 80)
print("Md. Faishal Ahmed Rudro")
print("Email: md.faishalrudro@gmail.com")
print("=" * 80)

# Load the dataset
df = pd.read_excel("/kaggle/input/testdatasetmain/test-dataset.xlsx")
print(f"Dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns.")

# Display initial dataset information
print("\n1. Dataset Overview:")
print("-" * 50)
print(f"Number of samples: {df.shape[0]}")
print(f"Number of features: {df.shape[1]}")
print(f"Missing values: {df.isnull().sum().sum()}")
print("\nTarget variable distribution (BP Status):")
if 'RESULT_STAT_BP' in df.columns:
    print(df['RESULT_STAT_BP'].value_counts())
    print("Percentage distribution:")
    print(df['RESULT_STAT_BP'].value_counts(normalize=True).apply(lambda x: f"{x*100:.2f}%"))
else:
    print("BP Status variable not found in dataset.")

# Identify variable types automatically
print("\n2. Variable Identification:")
print("-" * 50)
# Exclude identifier and obviously non-predictive columns
excluded_cols = [
    'Unnamed: 0', 'household_id', 'user_id', 'profile_name', 'father_name', 'mother_name',
    'birthday', 'TAG_NAME'
]

# Identify numerical and categorical features automatically
numeric_features = []
categorical_features = []

for col in df.columns:
    if col in excluded_cols or col in ['SYSTOLIC', 'DIASTOLIC', 'RESULT_STAT_BP']:
        continue
    
    if df[col].dtype in ['int64', 'float64']:
        numeric_features.append(col)
    elif df[col].dtype == 'object':
        if len(df[col].unique()) < df.shape[0] * 0.5:  # Avoid columns with too many unique values
            categorical_features.append(col)

print(f"Numerical features ({len(numeric_features)}):")
for i, feat in enumerate(numeric_features):
    print(f"{i+1}. {feat}")

print(f"\nCategorical features ({len(categorical_features)}):")
for i, feat in enumerate(categorical_features):
    print(f"{i+1}. {feat}")

print(f"\nTarget variables: SYSTOLIC, DIASTOLIC, RESULT_STAT_BP")

# Clean dataset for analysis - handle missing values
print("\n3. Data Preparation:")
print("-" * 50)
# For BP analysis
bp_df = df.dropna(subset=['RESULT_STAT_BP'])
print(f"Samples after removing missing BP status: {bp_df.shape[0]}")

# Missing value summary
print("\nMissing values per feature (top 10):")
missing_summary = df.isnull().sum().sort_values(ascending=False)
print(missing_summary[missing_summary > 0].head(10))

# Function to perform Chi-square test for categorical variables
def chi_square_test(df, feature, target, alpha=0.05):
    # Create contingency table
    contingency_table = pd.crosstab(df[feature], df[target])
    
    # Skip if not enough categories
    if contingency_table.shape[0] < 2 or contingency_table.shape[1] < 2:
        return feature, None, None, None, "Insufficient categories"
    
    # Perform chi-square test
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    
    # Check if more than 20% of cells have expected frequencies < 5
    expected_lt_5 = (expected < 5).sum()
    total_cells = expected.size
    percent_lt_5 = (expected_lt_5 / total_cells) * 100
    
    note = ""
    if percent_lt_5 > 20:
        note = f"Warning: {percent_lt_5:.1f}% of cells have expected frequency < 5"
    
    # Effect size (Cramer's V)
    n = contingency_table.sum().sum()
    min_dim = min(contingency_table.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0
    
    significance = "Significant" if p < alpha else "Not significant"
    return feature, p, contingency_table.shape, significance, note, cramers_v

# Function for numerical feature analysis with t-test or Mann-Whitney U test
def analyze_numerical_feature(df, feature, target, alpha=0.05):
    # Group data
    groups = []
    group_names = []
    
    for group_name in df[target].unique():
        group = df[df[target] == group_name][feature].dropna()
        if len(group) > 0:
            groups.append(group)
            group_names.append(group_name)
    
    if len(groups) < 2:
        return feature, None, "Insufficient data", None, None, None
    
    # If binary target and two groups
    if len(groups) == 2:
        # Use Mann-Whitney U test (non-parametric, doesn't assume normality)
        try:
            stat, p = mannwhitneyu(groups[0], groups[1])
            test_name = "Mann-Whitney U"
            
            # Calculate effect size (r = Z / sqrt(N))
            n1, n2 = len(groups[0]), len(groups[1])
            z = stat / np.sqrt((n1 * n2 * (n1 + n2 + 1)) / 6)
            effect_size = abs(z) / np.sqrt(n1 + n2)
            
        except Exception as e:
            # Fallback to t-test if Mann-Whitney fails
            stat, p = ttest_ind(groups[0], groups[1], equal_var=False)
            test_name = "Welch's t-test"
            
            # Calculate Cohen's d
            mean1, mean2 = np.mean(groups[0]), np.mean(groups[1])
            sd1, sd2 = np.std(groups[0], ddof=1), np.std(groups[1], ddof=1)
            pooled_sd = np.sqrt(((n1 - 1) * sd1**2 + (n2 - 1) * sd2**2) / (n1 + n2 - 2))
            effect_size = abs(mean1 - mean2) / pooled_sd
    else:
        # Use ANOVA for more than two groups
        stat, p = f_oneway(*groups)
        test_name = "ANOVA"
        
        # Calculate eta squared for ANOVA
        group_means = [np.mean(group) for group in groups]
        grand_mean = np.mean([val for group in groups for val in group])
        between_ss = sum(len(group) * (mean - grand_mean)**2 for group, mean in zip(groups, group_means))
        total_ss = sum((val - grand_mean)**2 for group in groups for val in group)
        effect_size = between_ss / total_ss if total_ss > 0 else 0
    
    # Calculate means for each group
    means = [group.mean() for group in groups]
    means_info = dict(zip(group_names, means))
    
    significance = "Significant" if p < alpha else "Not significant"
    return feature, p, significance, test_name, means_info, effect_size

print("\n4. Statistical Analysis:")
print("-" * 50)
print("\na) Categorical Variables Analysis with Chi-square Tests:")
print("   Analyzing relationship between categorical variables and BP status")

bp_categorical_results = []

for feature in categorical_features:
    if feature in bp_df.columns:
        result = chi_square_test(bp_df, feature, 'RESULT_STAT_BP')
        if result[1] is not None:
            bp_categorical_results.append(result)

# Sort by p-value
bp_categorical_results.sort(key=lambda x: x[1] if x[1] is not None else 1)

# Display results in tabular format
print("\n   Results sorted by statistical significance (p-value):")
print("   {:<30} {:<12} {:<25} {:<15} {:<10}".format("Feature", "p-value", "Table Shape", "Significance", "Effect Size"))
print("   " + "-" * 90)

for feature, p_val, shape, significance, note, effect_size in bp_categorical_results:
    if p_val is not None:
        print("   {:<30} {:<12.5f} {:<25} {:<15} {:<10.3f}".format(
            feature, p_val, str(shape), significance, effect_size))
        if note:
            print(f"   Note: {note}")

print("\nb) Numerical Variables Analysis:")
print("   Analyzing relationship between numerical variables and BP status")

bp_numerical_results = []

for feature in numeric_features:
    if feature in bp_df.columns:
        result = analyze_numerical_feature(bp_df, feature, 'RESULT_STAT_BP')
        if result[1] is not None:
            bp_numerical_results.append(result)

# Sort by p-value
bp_numerical_results.sort(key=lambda x: x[1] if x[1] is not None else 1)

# Display results in tabular format
print("\n   Results sorted by statistical significance (p-value):")
print("   {:<30} {:<12} {:<15} {:<15} {:<10}".format("Feature", "p-value", "Significance", "Test", "Effect Size"))
print("   " + "-" * 90)

for feature, p_val, significance, test_name, means_info, effect_size in bp_numerical_results:
    if p_val is not None:
        print("   {:<30} {:<12.5f} {:<15} {:<15} {:<10.3f}".format(
            feature, p_val, significance, test_name, effect_size))
        print("   Group means: " + ", ".join([f"{k}: {v:.2f}" for k, v in means_info.items()]))

# Feature importance using Random Forest for BP classification
print("\n5. Feature Importance Analysis:")
print("-" * 50)

def get_feature_importance(df, target, features):
    # Prepare data
    X = df[features].copy()
    y = df[target].copy()
    
    # Skip features with too many missing values
    valid_features = []
    for col in X.columns:
        if X[col].isnull().sum() / len(X) < 0.5:  # Skip if >50% missing
            valid_features.append(col)
        else:
            print(f"   Skipping feature '{col}' due to excessive missing values ({X[col].isnull().sum()} out of {len(X)})")
    
    X = X[valid_features]
    
    # Replace infinite values with NaN
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # Create numerical and categorical feature lists for appropriate preprocessing
    numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
    
    print(f"   Processing {len(numerical_cols)} numerical features and {len(categorical_cols)} categorical features")
    
    # Handle categorical features properly - convert to numeric
    for col in categorical_cols:
        # Use factorize to convert categorical values to numeric codes
        X[col] = pd.factorize(X[col])[0]
        # Fill NaNs with -1 (a distinct category)
        X[col] = X[col].fillna(-1)
    
    # Impute missing values in numerical columns
    for col in numerical_cols:
        if X[col].isnull().sum() > 0:
            X[col] = X[col].fillna(X[col].median())
    
    # Verify no NaNs remain
    if X.isnull().sum().sum() > 0:
        print(f"   Warning: {X.isnull().sum().sum()} missing values remain after imputation")
        # Fill any remaining NaNs with 0
        X = X.fillna(0)
    
    # Handle target variable
    if y.dtype == 'object':
        y = pd.factorize(y)[0]
    
    # Train a Random Forest classifier
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    
    try:
        print("   Fitting Random Forest model...")
        clf.fit(X, y)
        
        # Get feature importances
        importances = clf.feature_importances_
        
        # Create a DataFrame of feature importances
        feature_importance_df = pd.DataFrame({
            'Feature': X.columns,
            'Importance': importances
        }).sort_values('Importance', ascending=False)
        
        print(f"   Model fitting successful. Extracted importance for {len(X.columns)} features.")
        return feature_importance_df
    
    except Exception as e:
        print(f"   Error during model fitting: {e}")
        print("   Model input data types:", X.dtypes.value_counts())
        print("   First rows of processed data:")
        print(X.head(2))
        return pd.DataFrame(columns=['Feature', 'Importance'])

print("   Running Random Forest feature importance analysis for BP status prediction")
print("   This analysis helps identify which features are most predictive of BP status in a multivariate context\n")

# Prepare features for importance analysis
all_features = numeric_features + categorical_features
bp_features = [f for f in all_features if f in bp_df.columns]

try:
    # Get feature importance for BP
    bp_importance = get_feature_importance(bp_df, 'RESULT_STAT_BP', bp_features)
    
    if not bp_importance.empty:
        print("\n   Top 15 features by importance for BP status prediction:")
        print("   {:<30} {:<15}".format("Feature", "Importance Score"))
        print("   " + "-" * 45)
        for _, row in bp_importance.head(15).iterrows():
            print("   {:<30} {:<15.4f}".format(row['Feature'], row['Importance']))
    else:
        print("   Feature importance analysis did not produce valid results.")
        
except Exception as e:
    print(f"   Error in feature importance analysis: {e}")
    bp_importance = pd.DataFrame(columns=['Feature', 'Importance'])

# Correlation analysis for numerical features
print("\n6. Correlation Analysis for Numerical Features:")
print("-" * 50)
try:
    # Include only numeric features with <= 30% missing values
    valid_numeric = []
    for col in numeric_features:
        if col in bp_df.columns and bp_df[col].isnull().sum() / len(bp_df) <= 0.3:
            valid_numeric.append(col)
    
    numerical_df = bp_df[['SYSTOLIC', 'DIASTOLIC'] + valid_numeric].copy()
    
    # Impute missing values with median for correlation analysis
    for col in numerical_df.columns:
        if numerical_df[col].isnull().sum() > 0:
            numerical_df[col] = numerical_df[col].fillna(numerical_df[col].median())
    
    correlation_matrix = numerical_df.corr()
    
    print("   Key correlations with BP measurements:")
    systolic_corr = correlation_matrix['SYSTOLIC'].sort_values(ascending=False)
    diastolic_corr = correlation_matrix['DIASTOLIC'].sort_values(ascending=False)
    
    print("\n   Top correlations with SYSTOLIC BP:")
    print("   {:<30} {:<15}".format("Feature", "Correlation"))
    print("   " + "-" * 45)
    # Skip the self-correlation (SYSTOLIC with SYSTOLIC)
    for feature, corr in systolic_corr.iloc[1:7].items():
        print("   {:<30} {:<15.4f}".format(feature, corr))
    
    print("\n   Top correlations with DIASTOLIC BP:")
    print("   {:<30} {:<15}".format("Feature", "Correlation"))
    print("   " + "-" * 45)
    # Skip the self-correlation (DIASTOLIC with DIASTOLIC)
    for feature, corr in diastolic_corr.iloc[1:7].items():
        print("   {:<30} {:<15.4f}".format(feature, corr))
    
except Exception as e:
    print(f"   Error in correlation analysis: {e}")

print("\n7. Statistical Summary and Conclusions:")
print("-" * 50)
print("\nSignificant factors associated with BP status (p < 0.05):")

print("\nStatistically significant categorical factors:")
significant_categorical = [(feat, p_val, effect) for feat, p_val, _, sig, _, effect in bp_categorical_results 
                          if p_val is not None and p_val < 0.05]

if significant_categorical:
    print("   {:<30} {:<15} {:<15}".format("Feature", "p-value", "Effect Size"))
    print("   " + "-" * 60)
    for feat, p_val, effect in significant_categorical:
        print("   {:<30} {:<15.5f} {:<15.4f}".format(feat, p_val, effect))
else:
    print("   No statistically significant categorical features found")

print("\nStatistically significant numerical factors:")
significant_numerical = [(feat, p_val, effect) for feat, p_val, sig, _, _, effect in bp_numerical_results 
                        if p_val is not None and p_val < 0.05]

if significant_numerical:
    print("   {:<30} {:<15} {:<15}".format("Feature", "p-value", "Effect Size"))
    print("   " + "-" * 60)
    for feat, p_val, effect in significant_numerical:
        print("   {:<30} {:<15.5f} {:<15.4f}".format(feat, p_val, effect))
else:
    print("   No statistically significant numerical features found")

# Final conclusion
print("\nConclusion:")
if len(significant_categorical) > 0 or len(significant_numerical) > 0:
    print("   Statistical analysis reveals significant relationships between several factors and BP status.")
    print("   These factors appear to be strongly associated with blood pressure classification and could be")
    print("   important for predicting cardiovascular risk, similar to findings in the Kaunas cohort study.")
    
    if not bp_importance.empty:
        top_features = bp_importance['Feature'].head(5).tolist()
        print(f"\n   The most predictive features based on machine learning importance scores were:")
        print("   {:<30} {:<15}".format("Feature", "Importance"))
        print("   " + "-" * 45)
        for feat in top_features:
            importance = bp_importance[bp_importance['Feature'] == feat]['Importance'].values[0]
            print("   {:<30} {:<15.4f}".format(feat, importance))
    else:
        print("\n   Feature importance analysis did not produce valid results.")
else:
    print("   No statistically significant relationships were found between the examined factors and BP status.")
    print("   This differs from the Kaunas cohort study findings, which found strong relationships between")
    print("   childhood anthropometric measurements and adult cardiovascular risk factors.")

print("\nMethodological Note:")
print("   This analysis employed Chi-square tests for categorical variables and Mann-Whitney U/ANOVA tests")
print("   for numerical variables to identify statistically significant associations with BP status.")
print("   Effect sizes were calculated to quantify the strength of associations beyond statistical significance.")
print("   Additionally, Random Forest-based feature importance analysis was used to identify the most")
print("   predictive factors in a multivariate context, accounting for potential interactions between variables.")

Advanced Statistical Analysis for Cardiovascular Risk Factors
Md. Faishal Ahmed Rudro
Email: md.faishalrudro@gmail.com
Dataset loaded with 29999 rows and 34 columns.

1. Dataset Overview:
--------------------------------------------------
Number of samples: 29999
Number of features: 34
Missing values: 333065

Target variable distribution (BP Status):
RESULT_STAT_BP
Normal             11490
Prehypertension     7059
Mild High           3574
Low                 2704
Moderate High       1598
Severe High          630
High                 545
Name: count, dtype: int64
Percentage distribution:
RESULT_STAT_BP
Normal             41.63%
Prehypertension    25.58%
Mild High          12.95%
Low                 9.80%
Moderate High       5.79%
Severe High         2.28%
High                1.97%
Name: proportion, dtype: object

2. Variable Identification:
--------------------------------------------------
Numerical features (12):
1. age
2. is_poor
3. is_freedom_fighter
4. had_stroke
5. has_cardiovascu

In [7]:
df.columns

Index(['Unnamed: 0', 'household_id', 'total_income', 'union_name', 'user_id', 'profile_name', 'father_name', 'mother_name', 'birthday', 'age', 'gender', 'is_poor', 'is_freedom_fighter', 'had_stroke', 'has_cardiovascular_disease', 'disabilities_name', 'diabetic', 'profile_hypertensive', 'SYSTOLIC', 'DIASTOLIC', 'RESULT_STAT_BP', 'HEIGHT', 'WEIGHT', 'BMI', 'RESULT_STAT_BMI', 'SUGAR', 'TAG_NAME', 'RESULT_STAT_SUGAR', 'PULSE_RATE', 'RESULT_STAT_PR', 'SPO2', 'RESULT_STAT_SPO2', 'MUAC', 'RESULT_STAT_MUAC'], dtype='object')

In [8]:
total_columns = df.shape[1]

print(f"Total number of columns: {total_columns}")

Total number of columns: 34


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, ttest_ind, mannwhitneyu, f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Enhanced data loading and preprocessing
def load_and_preprocess_data(filepath):
    """
    Load and preprocess the dataset with advanced cleaning and transformation
    """
    # Load the dataset
    df = pd.read_excel(filepath)
    
    # Basic cleaning
    df.columns = df.columns.str.strip()
    
    # Remove columns with excessive missing values (>50%)
    missing_threshold = 0.5
    columns_to_drop = df.columns[df.isnull().mean() > missing_threshold]
    df = df.drop(columns=columns_to_drop)
    
    # Identify column types
    numeric_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = df.select_dtypes(include=['object']).columns.tolist()
    
    # Remove identifier and non-predictive columns
    excluded_cols = [
        'household_id', 'user_id', 'profile_name', 'father_name', 'mother_name',
        'birthday', 'TAG_NAME', 'Unnamed: 0'
    ]
    numeric_features = [col for col in numeric_features if col not in excluded_cols]
    categorical_features = [col for col in categorical_features if col not in excluded_cols]
    
    # Impute missing values
    # Numerical features: median imputation
    for col in numeric_features:
        df[col].fillna(df[col].median(), inplace=True)
    
    # Categorical features: mode imputation
    for col in categorical_features:
        df[col].fillna(df[col].mode()[0], inplace=True)
    
    return df, numeric_features, categorical_features

# Enhanced statistical analysis function
def comprehensive_feature_analysis(df, target_column, numeric_features, categorical_features):
    """
    Perform comprehensive statistical analysis of features
    """
    results = {
        'categorical': [],
        'numerical': [],
        'feature_importance': None
    }
    
    # Categorical feature analysis
    for feature in categorical_features:
        if feature == target_column:
            continue
        
        try:
            # Chi-square test
            contingency_table = pd.crosstab(df[feature], df[target_column])
            chi2, p_value, dof, expected = chi2_contingency(contingency_table)
            
            # Cramer's V effect size
            n = contingency_table.sum().sum()
            min_dim = min(contingency_table.shape) - 1
            cramer_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0
            
            results['categorical'].append({
                'feature': feature,
                'p_value': p_value,
                'cramer_v': cramer_v,
                'significant': p_value < 0.05
            })
        except Exception as e:
            print(f"Error analyzing categorical feature {feature}: {e}")
    
    # Numerical feature analysis
    for feature in numeric_features:
        if feature == target_column or feature in ['SYSTOLIC', 'DIASTOLIC']:
            continue
        
        try:
            # Group the data by target column
            groups = [group[feature].dropna() for name, group in df.groupby(target_column)]
            
            # ANOVA for multiple groups
            f_stat, p_value = f_oneway(*groups)
            
            # Effect size (Eta squared)
            group_means = [np.mean(group) for group in groups]
            grand_mean = np.mean([val for group in groups for val in group])
            between_ss = sum(len(group) * (mean - grand_mean)**2 for group, mean in zip(groups, group_means))
            total_ss = sum((val - grand_mean)**2 for group in groups for val in group)
            eta_squared = between_ss / total_ss if total_ss > 0 else 0
            
            results['numerical'].append({
                'feature': feature,
                'p_value': p_value,
                'eta_squared': eta_squared,
                'significant': p_value < 0.05
            })
        except Exception as e:
            print(f"Error analyzing numerical feature {feature}: {e}")
    
    # Feature importance using Random Forest
    try:
        # Prepare data for Random Forest
        X = df[numeric_features + categorical_features]
        
        # Encode categorical features
        le = LabelEncoder()
        for col in categorical_features:
            X[col] = le.fit_transform(X[col])
        
        # Encode target variable
        y = le.fit_transform(df[target_column])
        
        # Random Forest Classifier
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(X, y)
        
        # Feature importance
        importance_df = pd.DataFrame({
            'feature': X.columns,
            'importance': rf.feature_importances_
        }).sort_values('importance', ascending=False)
        
        results['feature_importance'] = importance_df
    
    except Exception as e:
        print(f"Error in feature importance analysis: {e}")
    
    return results

# Visualization functions
def plot_feature_importance(importance_df, title):
    """
    Create a bar plot of feature importances
    """
    plt.figure(figsize=(12, 6))
    sns.barplot(x='importance', y='feature', data=importance_df.head(15), 
                palette='viridis')
    plt.title(f'{title} - Top 15 Features by Importance')
    plt.xlabel('Importance Score')
    plt.ylabel('Features')
    plt.tight_layout()
    plt.savefig(f'{title.lower().replace(" ", "_")}_feature_importance.png')
    plt.close()

def plot_statistical_significance(results, analysis_type):
    """
    Create a bar plot of statistical significances
    """
    # Filter significant features
    significant_features = [
        (r['feature'], -np.log10(r['p_value']), r['p_value']) 
        for r in results[analysis_type] 
        if r['significant']
    ]
    
    if not significant_features:
        print(f"No significant {analysis_type} features found.")
        return
    
    # Sort by significance
    significant_features.sort(key=lambda x: x[1], reverse=True)
    
    plt.figure(figsize=(12, 6))
    plt.bar([f[0] for f in significant_features], 
            [f[1] for f in significant_features])
    plt.title(f'Statistical Significance of {analysis_type.capitalize()} Features')
    plt.xlabel('Features')
    plt.ylabel('-log10(p-value)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(f'{analysis_type}_significance.png')
    plt.close()

    # Print detailed results
    print(f"\nSignificant {analysis_type.capitalize()} Features:")
    print("{:<30} {:<15} {:<15}".format("Feature", "p-value", "Effect Size"))
    print("-" * 60)
    for feature, _, p_value in significant_features:
        effect_size = next(r['cramer_v'] if analysis_type == 'categorical' else r['eta_squared'] 
                           for r in results[analysis_type] 
                           if r['feature'] == feature)
        print("{:<30} {:<15.5f} {:<15.4f}".format(feature, p_value, effect_size))

# Main analysis function
def main_analysis(filepath):
    """
    Perform comprehensive data analysis
    """
    # Load and preprocess data
    df, numeric_features, categorical_features = load_and_preprocess_data(filepath)
    
    # Target variable analysis (assuming RESULT_STAT_BP is the target)
    target_column = 'RESULT_STAT_BP'
    
    print("Data Overview:")
    print(f"Total Samples: {len(df)}")
    print(f"Features: {len(numeric_features) + len(categorical_features)}")
    print(f"Target Variable Distribution:\n{df[target_column].value_counts(normalize=True)}")
    
    # Comprehensive feature analysis
    results = comprehensive_feature_analysis(
        df, target_column, numeric_features, categorical_features
    )
    
    # Visualizations
    if results['feature_importance'] is not None:
        plot_feature_importance(results['feature_importance'], 'BP Status')
    
    plot_statistical_significance(results, 'categorical')
    plot_statistical_significance(results, 'numerical')
    
    # Print summary
    print("\nAnalysis Summary:")
    if results['feature_importance'] is not None:
        print("\nTop 10 Most Important Features:")
        print(results['feature_importance'].head(10))
    
    return results

# Run the analysis
filepath = "/kaggle/input/testdatasetmain/test-dataset.xlsx"
analysis_results = main_analysis(filepath)

Data Overview:
Total Samples: 29999
Features: 14
Target Variable Distribution:
RESULT_STAT_BP
Normal             0.462982
Prehypertension    0.235308
Mild High          0.119137
Low                0.090136
Moderate High      0.053268
Severe High        0.021001
High               0.018167
Name: proportion, dtype: float64

Significant Categorical Features:
Feature                        p-value         Effect Size    
------------------------------------------------------------
union_name                     0.00000         0.1105         
RESULT_STAT_PR                 0.00000         0.1457         
gender                         0.00000         0.0930         
total_income                   0.00000         0.0515         

Significant Numerical Features:
Feature                        p-value         Effect Size    
------------------------------------------------------------
age                            0.00000         0.1540         
PULSE_RATE                     0.00000        

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# Enhanced feature importance analysis function
def comprehensive_feature_importance(filepath, target_column='RESULT_STAT_BP'):
    """
    Perform comprehensive feature importance analysis
    """
    # Load the dataset
    df = pd.read_excel(filepath)
    
    # Remove columns that are clearly not useful for prediction
    excluded_cols = [
        'Unnamed: 0', 'household_id', 'user_id', 'profile_name', 
        'father_name', 'mother_name', 'birthday', 'TAG_NAME'
    ]
    
    # Remove excluded columns
    df = df.drop(columns=[col for col in excluded_cols if col in df.columns])
    
    # Prepare features
    # Separate features and target
    features = df.columns.tolist()
    features.remove(target_column)
    
    # Prepare the dataframe for analysis
    X = df[features].copy()
    y = df[target_column]
    
    # Preprocessing pipeline
    preprocessor = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),  # Handles numerical missing values
        ('scaler', StandardScaler())  # Standardize features
    ])
    
    # Encode categorical variables
    le = LabelEncoder()
    for col in X.select_dtypes(include=['object']).columns:
        X[col] = le.fit_transform(X[col].astype(str))
    
    # Encode target variable
    y = le.fit_transform(y)
    
    # Prepare data for Random Forest
    X_processed = preprocessor.fit_transform(X)
    
    # Train Random Forest Classifier
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_processed, y)
    
    # Create feature importance DataFrame
    feature_importance = pd.DataFrame({
        'feature': features,
        'importance': rf.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # Visualization of all feature importances
    plt.figure(figsize=(20, 10))
    sns.barplot(x='importance', y='feature', data=feature_importance, palette='viridis')
    plt.title('Feature Importance for BP Status Prediction (All Features)', fontsize=16)
    plt.xlabel('Importance Score', fontsize=12)
    plt.ylabel('Features', fontsize=12)
    plt.tight_layout()
    plt.savefig('all_features_importance.png')
    plt.close()
    
    # Visualization of top 20 features
    plt.figure(figsize=(15, 8))
    sns.barplot(x='importance', y='feature', data=feature_importance.head(20), palette='rocket')
    plt.title('Top 20 Features by Importance for BP Status Prediction', fontsize=16)
    plt.xlabel('Importance Score', fontsize=12)
    plt.ylabel('Features', fontsize=12)
    plt.tight_layout()
    plt.savefig('top_20_features_importance.png')
    plt.close()
    
    # Print detailed results
    print("Comprehensive Feature Importance Analysis for BP Status")
    print("=" * 50)
    
    # Detailed print of all features
    print("\nFull Feature Importance Ranking:")
    print("{:<40} {:<15} {:<10}".format("Feature", "Importance Score", "Rank"))
    print("-" * 65)
    for rank, (_, row) in enumerate(feature_importance.iterrows(), 1):
        print("{:<40} {:<15.4f} {:<10}".format(row['feature'], row['importance'], rank))
    
    # Identify key feature groups
    def categorize_importance(importance):
        if importance > 0.05:
            return "High"
        elif importance > 0.02:
            return "Medium"
        elif importance > 0.01:
            return "Low"
        else:
            return "Very Low"
    
    feature_importance['importance_category'] = feature_importance['importance'].apply(categorize_importance)
    
    # Summary of feature importance categories
    print("\nFeature Importance Categories:")
    category_summary = feature_importance['importance_category'].value_counts()
    print(category_summary)
    
    # Detailed breakdown of top features by category
    print("\nTop Features by Importance Category:")
    for category in ['High', 'Medium', 'Low', 'Very Low']:
        top_features = feature_importance[feature_importance['importance_category'] == category].head(10)
        if not top_features.empty:
            print(f"\n{category} Importance Features:")
            print("{:<40} {:<15} {:<15}".format("Feature", "Importance Score", "Rank"))
            print("-" * 65)
            for rank, (_, row) in enumerate(top_features.iterrows(), 1):
                print("{:<40} {:<15.4f} {:<15}".format(row['feature'], row['importance'], rank))
    
    return feature_importance

# Run the analysis
filepath = "/kaggle/input/testdatasetmain/test-dataset.xlsx"
feature_importance_results = comprehensive_feature_importance(filepath)

Comprehensive Feature Importance Analysis for BP Status

Full Feature Importance Ranking:
Feature                                  Importance Score Rank      
-----------------------------------------------------------------
SYSTOLIC                                 0.4175          1         
DIASTOLIC                                0.3376          2         
RESULT_STAT_PR                           0.0758          3         
age                                      0.0706          4         
PULSE_RATE                               0.0267          5         
profile_hypertensive                     0.0234          6         
union_name                               0.0122          7         
RESULT_STAT_SPO2                         0.0081          8         
SPO2                                     0.0049          9         
RESULT_STAT_SUGAR                        0.0045          10        
total_income                             0.0045          11        
SUGAR                      